# 📊 Portfolio Risk & Optimization Analysis
**Auteur :** Brad Aymeric Feltre Kamanou — IÉSEG Master Data Management  
**Objectif :** Analyse du risque et optimisation d'un portefeuille multi-actifs via simulation Monte Carlo et optimisation scipy.

---
## 📋 Structure du notebook
1. [Importation des données](#1-importation)
2. [Analyse exploratoire](#2-exploration)
3. [Matrice de corrélation](#3-correlation)
4. [Rendements logarithmiques](#4-log-returns)
5. [Simulation Monte Carlo — Frontière Efficiente](#5-monte-carlo)
6. [Mesures de risque — VaR & Expected Shortfall](#6-var-es)
7. [Ratios de performance — Sharpe & Sortino](#7-ratios)
8. [Optimisation scipy — Portefeuille optimal](#8-optimisation)
9. [Comparaison Monte Carlo vs scipy](#9-comparaison)


## 1. Importation des données <a id='1-importation'></a>

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy.optimize import minimize

# Paramètres globaux
PERIODE = '504d'       # 2 ans de données
NB_JOURS = 252         # séances annuelles
RISK_FREE_RATE = 0.025 # taux sans risque (2.5%)
NB_SIMULATIONS = 10000 # Monte Carlo

### Composition du portefeuille initial
| Actif | Ticker | Pondération | Secteur |
|---|---|---|---|
| Microsoft | MSFT | 10% | Tech US |
| Apple | AAPL | 15% | Tech US |
| Air Liquide | AI.PA | 15% | Industrie EU |
| Or | GC=F | 50% | Matière première |

> La diversification sectorielle vise à réduire la corrélation entre actifs et limiter le risque global.

In [ ]:
# Téléchargement des données via yfinance (données en temps réel)
msft   = yf.download('MSFT',  period=PERIODE, auto_adjust=True)['Close']
aapl   = yf.download('AAPL',  period=PERIODE, auto_adjust=True)['Close']
airliq = yf.download('AI.PA', period=PERIODE, auto_adjust=True)['Close']
gold   = yf.download('GC=F',  period=PERIODE, auto_adjust=True)['Close']

# Consolidation dans un seul DataFrame
table_stocks = pd.concat([msft, aapl, airliq, gold], axis=1)
table_stocks = table_stocks.dropna()

print(f"Période : {table_stocks.index[0].date()} → {table_stocks.index[-1].date()}")
print(f"Nombre de séances : {len(table_stocks)}")
table_stocks.tail()

## 2. Analyse exploratoire <a id='2-exploration'></a>

In [ ]:
# Rendements moyens journaliers par actif (en %)
print("Rendements moyens journaliers (%) :")
print((table_stocks.pct_change(fill_method=None).mean() * 100).round(4))

In [ ]:
# Évolution des prix sur la période
fig = px.line(table_stocks, title="Évolution des prix sur 2 ans",
              labels={'value': 'Prix', 'Date': 'Date', 'variable': 'Actif'})
fig.update_layout(hovermode='x unified')
fig.show()

## 3. Matrice de corrélation <a id='3-correlation'></a>
> La corrélation mesure comment deux actifs évoluent ensemble.  
> **+1** = évoluent identiquement | **0** = indépendants | **-1** = évoluent en sens opposé

In [ ]:
# Calcul de la matrice de corrélation
correlation_matrix = table_stocks.pct_change(fill_method=None).dropna().corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm',
            fmt='.2f', linewidths=0.5, vmin=-1, vmax=1)
plt.title('Matrice de corrélation des rendements journaliers')
plt.tight_layout()
plt.show()

print("\nInterprétation :")
print(f"  MSFT / AAPL  : {correlation_matrix.loc['MSFT','AAPL']:.2f} → corrélation modérée (même secteur tech)")
print(f"  AAPL / GC=F  : {correlation_matrix.loc['AAPL','GC=F']:.2f} → quasi décorrélés (diversification efficace)")
print(f"  MSFT / GC=F  : {correlation_matrix.loc['MSFT','GC=F']:.2f} → quasi décorrélés")

## 4. Rendements logarithmiques <a id='4-log-returns'></a>
> Les rendements logarithmiques sont préférés aux rendements simples car ils sont additifs dans le temps et plus adaptés aux calculs de risque.

In [ ]:
# Calcul des rendements logarithmiques
log_ret = np.log(table_stocks / table_stocks.shift(1)).dropna()

print("Rendements logarithmiques moyens journaliers :")
print(log_ret.mean().round(6))
print(f"\nNombre d'observations : {len(log_ret)}")
log_ret.tail()

## 5. Simulation Monte Carlo — Frontière Efficiente <a id='5-monte-carlo'></a>
> Le Monte Carlo génère **10 000 portefeuilles** avec des pondérations aléatoires et calcule le rendement/risque de chacun.  
> Chaque point sur le graphique = un portefeuille possible.

In [ ]:
# Initialisation des structures de stockage
all_weights  = np.zeros((NB_SIMULATIONS, len(table_stocks.columns)))
retour_list  = np.zeros(NB_SIMULATIONS)
risk_list    = np.zeros(NB_SIMULATIONS)
sharpe_list  = np.zeros(NB_SIMULATIONS)

# Simulation
np.random.seed(42)  # reproductibilité
for ind in range(NB_SIMULATIONS):
    # 1. Poids aléatoires + renormalisation (somme = 1)
    weights = np.random.random(len(table_stocks.columns))
    weights = weights / np.sum(weights)
    all_weights[ind, :] = weights

    # 2. Rendement annualisé (produit scalaire w · μ × 252)
    retour_list[ind] = np.sum(log_ret.mean() * weights) * NB_JOURS

    # 3. Risque annualisé (forme quadratique √(wᵀΣw))
    risk_list[ind] = np.sqrt(np.dot(weights.T,
                              np.dot(log_ret.cov() * NB_JOURS, weights)))

    # 4. Ratio de Sharpe
    sharpe_list[ind] = retour_list[ind] / risk_list[ind]

print(f"✅ {NB_SIMULATIONS} portefeuilles simulés")
print(f"Sharpe max (Monte Carlo) : {sharpe_list.max():.4f}")

In [ ]:
# Visualisation de la Frontière Efficiente
df_mc = pd.DataFrame({
    'Return [%]': retour_list * 100,
    'Risk [%]'  : risk_list * 100,
    'Sharpe'    : sharpe_list
})

# Portefeuille optimal Monte Carlo
best_idx = sharpe_list.argmax()

fig = px.scatter(df_mc, x='Risk [%]', y='Return [%]', color='Sharpe',
                 color_continuous_scale='Viridis',
                 title='Frontière Efficiente — 10 000 Portefeuilles (Monte Carlo)',
                 labels={'Risk [%]': 'Risque (%)', 'Return [%]': 'Rendement (%)'})

# Marqueur portefeuille optimal
fig.add_scatter(x=[risk_list[best_idx] * 100],
                y=[retour_list[best_idx] * 100],
                mode='markers',
                marker=dict(size=12, color='red', symbol='star'),
                name='Max Sharpe (Monte Carlo)')
fig.show()

## 6. Mesures de risque — VaR & Expected Shortfall <a id='6-var-es'></a>
> - **VaR (99%)** : perte maximale journalière avec 99% de confiance  
> - **Expected Shortfall (99%)** : moyenne des pertes au-delà de la VaR (pire 1% des cas)

In [ ]:
# Poids équipondérés pour le calcul de VaR
weights_eq = np.array([1/4, 1/4, 1/4, 1/4])
portfolio_returns = log_ret.dot(weights_eq)

# VaR Historique (99%)
confidence_level = 0.01
historical_var = np.percentile(portfolio_returns, confidence_level * 100)

# Expected Shortfall (99%)
expected_shortfall = portfolio_returns[portfolio_returns <= historical_var].mean()

print(f"VaR historique (99%)      : {abs(historical_var):.4f} ({abs(historical_var)*100:.2f}%)")
print(f"Expected Shortfall (99%)  : {abs(expected_shortfall):.4f} ({abs(expected_shortfall)*100:.2f}%)")
print(f"\nInterprétation : Dans 99% des cas, la perte journalière ne dépasse pas {abs(historical_var)*100:.2f}%.")
print(f"Dans le pire 1% des cas, la perte moyenne est de {abs(expected_shortfall)*100:.2f}%.")

In [ ]:
# Visualisation de la distribution des rendements
plt.figure(figsize=(12, 6))
plt.hist(portfolio_returns, bins=60, density=True, alpha=0.7,
         color='steelblue', edgecolor='white', label='Rendements journaliers')

# Courbe de densité
from scipy.stats import gaussian_kde
kde = gaussian_kde(portfolio_returns)
x_range = np.linspace(portfolio_returns.min(), portfolio_returns.max(), 300)
plt.plot(x_range, kde(x_range), color='navy', linewidth=2)

# Seuils VaR et ES
plt.axvline(historical_var, color='red', linestyle='--', linewidth=2,
            label=f'VaR (99%) : {abs(historical_var):.4f}')
plt.axvline(expected_shortfall, color='green', linestyle=':', linewidth=2,
            label=f'ES (99%)  : {abs(expected_shortfall):.4f}')

plt.title('Distribution des rendements du portefeuille avec VaR et ES (99%)')
plt.xlabel('Rendements journaliers')
plt.ylabel('Densité')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Ratios de performance — Sharpe & Sortino <a id='7-ratios'></a>
> - **Sharpe** = (Rendement - Taux sans risque) / Volatilité totale  
> - **Sortino** = (Rendement - Taux sans risque) / Volatilité **baissière** uniquement  
> Le Sortino est plus pertinent car il ne pénalise pas la volatilité haussière.

In [ ]:
# Rendements journaliers par actif
rendements = table_stocks.pct_change(fill_method=None).dropna()

# Ratio de Sortino par actif
def sortino(returns, seuil=0, risk_free_rate=RISK_FREE_RATE):
    """
    Calcule le ratio de Sortino d'un actif.
    returns          : série de rendements journaliers
    seuil            : rendement minimum acceptable (défaut = 0)
    risk_free_rate   : taux sans risque annualisé
    """
    rp           = returns.mean() * NB_JOURS
    neg_ret      = returns[returns < seuil]
    downside_vol = neg_ret.std() * np.sqrt(NB_JOURS)
    return round((rp - risk_free_rate) / downside_vol, 2)

print("Ratios de Sortino par actif :")
print("-" * 35)
for col in table_stocks.columns:
    ratio = sortino(rendements[col])
    interpretation = "✅ bon" if ratio > 1 else "⚠️ faible" if ratio > 0 else "❌ négatif"
    print(f"  {col:<8} : {ratio:>6} {interpretation}")

print("\nNote : Sortino > 1 = le rendement compense bien le risque baissier")

## 8. Optimisation scipy — Portefeuille optimal <a id='8-optimisation'></a>
> Le Monte Carlo approxime l'optimum par tirage aléatoire.  
> **scipy** calcule **exactement** les poids qui maximisent le Sharpe sous contraintes :
> - Contrainte d'égalité : Σ poids = 1
> - Contraintes d'inégalité : 0 ≤ poids ≤ 1 (pas de vente à découvert)

In [ ]:
from scipy.optimize import minimize

def neg_sharpe(weights):
    """
    Fonction objectif : retourne le Sharpe négatif.
    scipy minimise → minimiser(-Sharpe) = maximiser(Sharpe)
    """
    retour = np.sum(log_ret.mean() * weights) * NB_JOURS
    risk   = np.sqrt(np.dot(weights.T, np.dot(log_ret.cov() * NB_JOURS, weights)))
    return -(retour / risk)

# Contraintes et bornes
contraintes = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}  # somme = 1
bornes      = [(0, 1) for _ in range(len(table_stocks.columns))]  # poids ≥ 0

# Lancement de l'optimisation
resultat = minimize(
    neg_sharpe,
    x0=[0.25, 0.25, 0.25, 0.25],  # point de départ : poids égaux
    method='SLSQP',
    bounds=bornes,
    constraints=contraintes
)

poids_optimaux = resultat.x

# Affichage des résultats
print("=" * 40)
print("PORTEFEUILLE OPTIMAL (scipy)")
print("=" * 40)
for actif, poids in zip(table_stocks.columns, poids_optimaux):
    print(f"  {actif:<8} : {poids*100:>6.1f}%")
print("-" * 40)
print(f"  Sharpe optimal : {-resultat.fun:.4f}")
print(f"  Convergence    : {'✅ Oui' if resultat.success else '❌ Non'}")
print(f"  Itérations     : {resultat.nit}")

## 9. Comparaison Monte Carlo vs scipy <a id='9-comparaison'></a>
> Validation croisée des deux méthodes sur les **mêmes données**.

In [ ]:
# Recalcul du Sharpe Monte Carlo sur les mêmes log_ret
best_weights_mc = all_weights[sharpe_list.argmax()]
retour_mc = np.sum(log_ret.mean() * best_weights_mc) * NB_JOURS
risk_mc   = np.sqrt(np.dot(best_weights_mc.T,
                    np.dot(log_ret.cov() * NB_JOURS, best_weights_mc)))
sharpe_mc = retour_mc / risk_mc

# Tableau comparatif
print("=" * 55)
print(f"{'':20} {'Monte Carlo':>15} {'scipy':>15}")
print("=" * 55)
print(f"{'Sharpe':20} {sharpe_mc:>15.4f} {-resultat.fun:>15.4f}")
print(f"{'Rendement (%)':20} {retour_mc*100:>15.2f} {np.sum(log_ret.mean()*poids_optimaux)*NB_JOURS*100:>15.2f}")
print(f"{'Risque (%)':20} {risk_mc*100:>15.2f} {np.sqrt(np.dot(poids_optimaux.T, np.dot(log_ret.cov()*NB_JOURS, poids_optimaux)))*100:>15.2f}")
print("-" * 55)
print("Poids :")
for actif, w_mc, w_sp in zip(table_stocks.columns, best_weights_mc, poids_optimaux):
    print(f"  {actif:<8}  {w_mc*100:>13.1f}%  {w_sp*100:>13.1f}%")
print("=" * 55)
print("\nConclusion :")
print(f"  scipy est {'meilleur' if -resultat.fun > sharpe_mc else 'similaire'} au Monte Carlo")
print(f"  Différence de Sharpe : {abs(-resultat.fun - sharpe_mc):.4f}")
print("  → Les deux méthodes convergent vers la même allocation Or + AAPL")

In [ ]:
# Visualisation finale : frontière + portefeuille optimal scipy
fig = px.scatter(df_mc, x='Risk [%]', y='Return [%]', color='Sharpe',
                 color_continuous_scale='Viridis',
                 title='Frontière Efficiente — Monte Carlo vs scipy',
                 opacity=0.6)

# Portefeuille optimal Monte Carlo
fig.add_scatter(x=[risk_mc * 100],
                y=[retour_mc * 100],
                mode='markers',
                marker=dict(size=12, color='orange', symbol='star'),
                name=f'Monte Carlo (Sharpe={sharpe_mc:.2f})')

# Portefeuille optimal scipy
r_sp = np.sum(log_ret.mean() * poids_optimaux) * NB_JOURS
v_sp = np.sqrt(np.dot(poids_optimaux.T, np.dot(log_ret.cov()*NB_JOURS, poids_optimaux)))
fig.add_scatter(x=[v_sp * 100],
                y=[r_sp * 100],
                mode='markers',
                marker=dict(size=14, color='red', symbol='star'),
                name=f'scipy optimal (Sharpe={-resultat.fun:.2f})')

fig.update_layout(hovermode='closest')
fig.show()

---
## 📌 Conclusions

| Indicateur | Valeur |
|---|---|
| VaR historique (99%) | ~2.4% |
| Expected Shortfall (99%) | ~2.9% |
| Sharpe optimal (scipy) | ~1.54 |
| Allocation optimale | ~57% Or, ~39% AAPL, ~4% Air Liquide, ~0% MSFT |

**Points clés :**
- L'Or (~57%) domine le portefeuille optimal grâce à sa faible corrélation avec les actions tech
- MSFT est éliminé par l'algorithme — AAPL offre un meilleur ratio rendement/risque sur la période
- Monte Carlo et scipy convergent vers la même allocation → résultat robuste

## 🔮 Améliorations futures
- [ ] Ajouter la VaR paramétrique et Monte Carlo pour comparer 3 méthodes
- [ ] Backtesting de la VaR (nombre de dépassements observés)
- [ ] Contraintes de poids (min 5%, max 40% par actif)
- [ ] Déployer une interface Streamlit interactive
- [ ] Étendre à un portefeuille de crédit bancaire (NPL, coût du risque)
